## HINDI SENTIMENT ANALYSIS

In [3]:
import pandas as pd 
import numpy as np 

In [4]:
path = r"data\hindi.csv"

In [5]:
df = pd.read_csv(r"D:\NLP Mini Project\data\hindi.csv")

In [6]:
df.head(5)

,Label,Sentence
0,neutral,आप अपने हाथ भरे होंगे।
1,neutral,कि मैंने किया। कि मैंने किया।
2,neutral,तो चलो अपने कर्तव्यों के बारे में थोड़ा बात करें।
3,surprise,मेरे कर्तव्य? ठीक है.
4,neutral,"अब आप एक पूरे डिवीजन का नेतृत्व करेंगे, तो आप ..."


In [7]:
X = df["Sentence"]
y= df["Label"]

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

vectorizer = TfidfVectorizer(token_pattern=r"\S+")
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train_tfidf, y_train)


print(len(vectorizer.get_feature_names_out()))
print(classification_report(y_test, model.predict(X_test_tfidf), zero_division=0))

6789
              precision    recall  f1-score   support

       anger       0.28      0.30      0.29       186
     disgust       0.09      0.21      0.13        47
        fear       0.06      0.14      0.09        49
         joy       0.41      0.33      0.37       283
     neutral       0.68      0.47      0.56       744
     sadness       0.18      0.34      0.24       112
    surprise       0.33      0.40      0.36       179

    accuracy                           0.39      1600
   macro avg       0.29      0.31      0.29      1600
weighted avg       0.47      0.39      0.42      1600



In [12]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate
import re

def to_hinglish(text):
    # Devanagari -> ITRANS, then lowercase and simplify into casual typing
    s = transliterate(text, sanscript.DEVANAGARI, sanscript.ITRANS)
    s = s.replace("|", " ").replace(".N", "n").replace("RRi", "ri")
    s = s.replace("M", "n").replace("~N", "n").replace("N", "n")
    s = s.replace("aa", "a").replace("A", "a").replace("I", "i").replace("U", "u")
    s = s.lower()
    s = re.sub(r"(.)\1{2,}", r"\1\1", s)   # cap repeated letters at 2
    s = re.sub(r"[?.,!\"']", " ", s)
    return re.sub(r"\s+", " ", s).strip()

X_test_hinglish = X_test.apply(to_hinglish)
print(X_test_hinglish.head(5).tolist())

['hey gunther hi main matalaba hai ki apa eka tarikha ke upara dhvani vale adami ke satha bahara ja rahe hain', 'kya chata men khi daki ke lie kya hua', 'han achchi taraha se eka bara phira nahin dankane ka bhugatana karata hai main kevala wish you hadn t been on the toilet', 'usane meri banha ko chu liya', 'main tumhen suna lekina durbhagyavasha meri kanpani mujhe videshon men sthanantarita kara rahi hai']


In [13]:
X_test_hing_tfidf = vectorizer.transform(X_test_hinglish)
print(classification_report(y_test, model.predict(X_test_hing_tfidf), zero_division=0))

              precision    recall  f1-score   support

       anger       0.00      0.00      0.00       186
     disgust       0.00      0.00      0.00        47
        fear       0.00      0.00      0.00        49
         joy       0.16      0.09      0.12       283
     neutral       0.46      0.85      0.59       744
     sadness       0.00      0.00      0.00       112
    surprise       0.22      0.06      0.09       179

    accuracy                           0.42      1600
   macro avg       0.12      0.14      0.11      1600
weighted avg       0.26      0.42      0.31      1600



In [15]:
import pandas as pd

X_train_hinglish = X_train.apply(to_hinglish)

X_train_mix = pd.concat([X_train, X_train_hinglish])
y_train_mix = pd.concat([y_train, y_train])

vec_mix = TfidfVectorizer(token_pattern=r"\S+")
Xtr = vec_mix.fit_transform(X_train_mix)

model_mix = LogisticRegression(max_iter=1000, class_weight="balanced")
model_mix.fit(Xtr, y_train_mix)

from sklearn.metrics import f1_score

def macro_f1(mdl, vec, X, y):
    return f1_score(y, mdl.predict(vec.transform(X)), average="macro", zero_division=0)

results = pd.DataFrame({
    "Hindi test":    [macro_f1(model, vectorizer, X_test, y_test),
                      macro_f1(model_mix, vec_mix, X_test, y_test)],
    "Hinglish test": [macro_f1(model, vectorizer, X_test_hinglish, y_test),
                      macro_f1(model_mix, vec_mix, X_test_hinglish, y_test)],
}, index=["Trained on Hindi only", "Trained on Hindi + Hinglish"])
print(results.round(3))

                             Hindi test  Hinglish test
Trained on Hindi only             0.289          0.114
Trained on Hindi + Hinglish       0.283          0.267


In [17]:
def add_noise(text, level, seed):
    rng = random.Random(seed * 1000003 + sum(map(ord, text)))
    out = []
    for word in text.split():
        if rng.random() < level:
            pat, reps = rng.choice(SWAPS)
            if pat in word:
                word = word.replace(pat, rng.choice(reps), 1)
            elif len(word) > 3:
                i = rng.randrange(1, len(word))
                word = word[:i] + word[i+1:]        # drop a letter
            elif len(word) > 1:
                word = word + word[-1]              # stretch a short word: hai -> haii
        out.append(word)
    return " ".join(out)

levels = {"None": 0.0, "Low": 0.3, "Medium": 0.6, "High": 1.0}
seeds = [1, 2, 3, 4, 5]

def changed_fraction(a, b):
    wa = [w for s in a for w in s.split()]
    wb = [w for s in b for w in s.split()]
    return sum(x != y for x, y in zip(wa, wb)) / len(wa)

rows = {}
for name, lvl in levels.items():
    scores, fracs = [], []
    for sd in seeds:
        Xn = X_test_hinglish.apply(lambda t: add_noise(t, lvl, sd))
        scores.append(macro_f1(model_mix, vec_mix, Xn, y_test))
        fracs.append(changed_fraction(X_test_hinglish, Xn))
    rows[name] = [f"{np.mean(fracs):.0%}", f"{np.mean(scores):.3f} ± {np.std(scores):.3f}"]

print(pd.DataFrame(rows, index=["% words changed", "macro F1 (5 runs)"]))

                            None            Low         Medium           High
% words changed               0%            30%            60%           100%
macro F1 (5 runs)  0.267 ± 0.000  0.246 ± 0.009  0.224 ± 0.004  0.140 ± 0.003


## NORMALIZATION

In [18]:
import difflib
from collections import Counter

vocab_counts = Counter(w for s in X_train_hinglish for w in s.split())
vocab = [w for w, _ in vocab_counts.most_common()]
vocab_set = set(vocab)
by_len = {}                                   
for w in vocab:
    by_len.setdefault(len(w), []).append(w)

def normalize_word(w, cutoff=0.75):
    if w in vocab_set:
        return w
    cands = [c for L in (len(w)-1, len(w), len(w)+1) for c in by_len.get(L, [])]
    m = difflib.get_close_matches(w, cands, n=1, cutoff=cutoff)
    return m[0] if m else w

def normalize(text):
    return " ".join(normalize_word(w) for w in text.split())

rows = {}
for name, lvl in levels.items():
    noisy, fixed = [], []
    for sd in seeds:
        Xn = X_test_hinglish.apply(lambda t: add_noise(t, lvl, sd))
        noisy.append(macro_f1(model_mix, vec_mix, Xn, y_test))
        fixed.append(macro_f1(model_mix, vec_mix, Xn.apply(normalize), y_test))
    rows[name] = [f"{np.mean(noisy):.3f}", f"{np.mean(fixed):.3f}"]

print(pd.DataFrame(rows, index=["Noisy", "Normalized"]))

             None    Low Medium   High
Noisy       0.267  0.246  0.224  0.140
Normalized  0.275  0.268  0.260  0.244
